In [ ]:
BATCH_SIZE = 32


In [1]:
import sys
sys.path.append('../../../')
from models.value_cnn import ValueCNNDecentralized, ValueCNNCentralized
from models.cnn import CNNDecentralized
import torch
from tadd_helpers.env_functions import State
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

--- PyTorch is configured to use: cpu ---


In [ ]:
def get_decentralized_reward_when_picked(picked: bool, picker_index, picker_pos, all_agents) -> dict[int, float]:
    """Return decentralized reward of all agents

    Args:
        picked (bool): Whether an agent has picked an apple
        picker_index: _description_
        picker_pos: _description_
        all_agents: dict[int, tuple]: Mapping from agent index to (x, y) position
    """
    res: dict[int, float] = {}
    
    if not picked:
        for agent_idx in all_agents.keys():
            res[agent_idx] = 0.0
        return res

    res[picker_index] = -1

    # Calculate distances from ALL agents to the picker
    all_agent_positions = np.array(list(all_agents.values()))
    distances = np.linalg.norm(all_agent_positions - np.array(picker_pos), axis=1)
    sum_of_distances = np.sum(distances)
    
    for agent_idx in all_agents.keys():
        if agent_idx == picker_index:
            continue
        agent_pos = all_agents[agent_idx]
        agent_distance = np.linalg.norm(np.array(agent_pos) - np.array(picker_pos))
        if sum_of_distances == 0:
            res[agent_idx] = 0.0
        else:
            res[agent_idx] = 2 * agent_distance / sum_of_distances
    return res

def get_picker_index_and_pos_from_state(state: State):
    """Extract the picker index and position from the state. 

    Args:
        state (State): The current environment state
    Returns:
        picked: bool: Whether an agent has picked an apple
        picker_index (int): The index of the picker agent
        picker_pos (tuple): The (x, y) position of the picker agent 
    """
    picked = False
    picker_index = -1
    picker_pos = (-1, -1)
    for agent_idx, agent_pos in state._agents.items():
        # check if the agent_pos is in the apples nd array
        if state._apples[agent_pos] >= 1:
            picked = True
            picker_index = agent_idx
            picker_pos = agent_pos
            break
    return picked, picker_index, picker_pos

def get_decentralized_reward_from_state(state: State) -> dict[int, float]:
    """Calculate the decentralized reward for the self-agent based on the state.

    Args:
        state (State): The current environment state
    Returns:
        dict[int, float]: The decentralized rewards for all agents
    """
    picked, picker_index, picker_pos = get_picker_index_and_pos_from_state(state)
    all_agents = state._agents
    rewards = get_decentralized_reward_when_picked(picked, picker_index, picker_pos, all_agents)
    return rewards

In [ ]:
decen_path = "reward_model_6x6_2a"
decen_model_paths = []
for i in range(2):
    decen_model_paths.append(f"{decen_path}/model_{i}.pt")

['decentralized_model6x6_agents2/decentralized_value_cnn_agent_0.pt', 'decentralized_model6x6_agents2/decentralized_value_cnn_agent_1.pt']


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
decen_cnns: list[CNNDecentralized] = []
for i in range(len(decen_model_paths)):
    model_path = decen_model_paths[i]
    cnn = CNNDecentralized(6, 6, 0.0001, 128, 2, [16, 32], 3)
    
    state_dict = torch.load(model_path, map_location=device)
    cnn.load_state_dict(state_dict)
    # --------------------
    
    # 3. Explicitly move the entire model to the chosen device
    cnn.to(device)

    cnn.eval()
    decen_cnns.append(cnn)


ValueCNNCentralized(
  (conv_layers): ModuleList(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (mlp_head): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
  (target_net): CNNCentralized(
    (conv_layers): ModuleList(
      (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): Max

In [5]:
states: list[State] = []
states_file_path = "centralized_model6x6_agents2/trained_states_centralized.pkl"
with open(states_file_path, "rb") as f:
    states = pickle.load(f)
print(states[:3])

[State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]]), _agents={0: (3, 4), 1: (2, 4)}, name='Empty State'), State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]]), _agents={0: (3, 4), 1: (np.int64(1), np.int64(4))}, name='Empty State'), State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0]]), _agents={0: (3, 4), 1: (np.int64(1), np.int64(4))}, name='Empty State')]


In [ ]:
print(len(states))
states_to_eval = states

800000


In [ ]:
all_results = []
irs = {0: [], 1: []}
for t, state in tqdm(enumerate(states_to_eval), total=len(states_to_eval), desc="Evaluating states"):
    val_d = []
    for i in range(2):
        cnn = decen_cnns[i]
        raw_dict = cnn.state_to_raw_dict(state)
        ir_pred = cnn.get_model_reward_prediction_from_raw(raw_dict, agent_pos=state.agent_position(i))
        irs[i].append(ir_pred)
    true_rewards = get_decentralized_reward_from_state(state)

    result_info = {
        "timestep": (t / 2),
        "agent 0 error": val_d_team,
        "state_object": state
    }
    all_results.append(result_info)

10it [00:00, 298.46it/s]


In [10]:
df = pd.DataFrame(all_results)
print(df.head())

   timestep  centralized_team_value  decentralized_team_value     error  \
0       0.0               25.828159                 26.031549  0.203389   
1       0.5               25.774891                 26.388330  0.613440   
2       1.0               27.340803                 27.258855  0.081948   
3       1.5               27.311968                 27.621363  0.309395   
4       2.0               26.131105                 27.006741  0.875635   

   centralized - decentralized  \
0                    -0.203389   
1                    -0.613440   
2                     0.081948   
3                    -0.309395   
4                    -0.875635   

                                        state_object  
0  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
1  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
2  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
3  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
4  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  


In [9]:
with open("predicted_trajectory_values_6x6_agents2.pkl", "wb") as f:
    pickle.dump(df, f)